In [5]:
# test forecast
import sys
import os
import numpy as np
from scipy import stats
import json

from itertools import combinations

# Navigate from test/ to src/ using relative path
notebook_dir = os.path.dirname(os.path.abspath('.'))
src_path = os.path.join(notebook_dir,  'src')
sys.path.insert(0, src_path)

from common_lib import CommonLib
from stats_lib import OccurencesStats

######################
def get_occurrences_stats(evidenze):
    occurrences = evidenze.get('occurrences', {})
    
    limits = [] * len(occurrences)
    for key, class_data in occurrences.items(): 
        k = class_data.get("k", int(key))
        l_class = {}
        l_class["sample"] = k
        l_class["class"] = [{}] * len(class_data)
        for c_key, c_value in class_data.items():
            #print(f"  {c_key}")
            desc = c_value.get("description", "")
            measures = c_value.get("measures", {})
            values = measures.get("values", [])
            
            #print(f"\nClasse k={k}: {c_value} {desc}")
            s_limit = {}
            s_limit["k"] = c_key
            s_limit["bounds"] = [{}] * 4 
            for measure in values:
                m_id = measure.get("id", "")
                m_type = measure.get("type", "")
                m_mean = measure.get("mean", [])
                m_std = measure.get("std", [])
                m_skew = measure.get("skewness", [])
                m_q1 = measure.get("q1", [])
                m_q3 = measure.get("q3", [])
                
                # print(f"  {m_type}:")
                # print(f"    mean    : {m_mean}")
                # print(f"    std     : {m_std}")
                # print(f"    skewness: {m_skew}")
                # print(f"    q1      : {m_q1}")
                # print(f"    q3      : {m_q3}")

                simm = measure.get("simmetrica", {})
                asimm = measure.get("asimmetrica", {})
                
                bounds = {}
                t = abs(m_skew)
                match t:
                    case t if t < 0.5:
                        #print(f"simmetrica   {t}: min={simm.get('min')}, max={simm.get('max')}, coverage={simm.get('coverage')}%")
                        bounds = {"id": m_id, "min": simm.get('min'), "max": simm.get('max')}
                    case t if t > 1.0:
                        #print(f"asimmetrica  {t}: min={asimm.get('min')}, max={asimm.get('max')}, coverage={asimm.get('coverage')}%")
                        bounds = {"id": m_id, "min": asimm.get('min'), "max": asimm.get('max')}
                    case _:
                        if simm.get('coverage') >= asimm.get('coverage'):
                            bounds = {"id": m_id, "min": simm.get('min'), "max": simm.get('max')}
                            #print(f"simmetrica   {t}: min={simm.get('min')}, max={simm.get('max')}, coverage={simm.get('coverage')}%")
                        else:
                            #print(f"asimmetrica  {t}: min={asimm.get('min')}, max={asimm.get('max')}, coverage={asimm.get('coverage')}%")
                            bounds = {"id": m_id, "min": asimm.get('min'), "max": asimm.get('max')}
                
                #print(f"    bounds   : min={bounds.get('min')}, max={bounds.get('max')}")
                s_limit["bounds"][int(m_id)] = bounds
                
            #print(f"s_limit: {s_limit}")
        
            l_class['class'][int(c_key)-1] = s_limit
            #print(f"l_class: {l_class}")
            
        limits = limits + [l_class]
        

    return limits

def get_base_occurrences(base_combinations, sample):
    occurrences = [[] * 4] * 6
    for k in range(1,7):
        o = ostats.single_combination_occurrences(base_combinations[k-1], sample)
        #print(f"Occurrences for {base}: {o} , k = {k}")
        tot = np.sum(o)
        nz = np.count_nonzero(o)
        z = len(base_combinations[k-1]) - nz
        m = np.max(o)
        #print(f"Total occurrences: {tot} - Non-zero occurrences: {nz} - Zero occurrences: {z} - Max occurrences: {m}")
        occurrences[k-1] = [tot, nz, z, m]
        
    return occurrences

def check_occurrences(base_combinations, history_data, limits):

    for o in limits:
        sample_size = int(o.get('sample'))
        sample = history_data[:sample_size] 
        occurrences = get_base_occurrences(base_combinations, sample)
        # print(f"Occurrences: {occurrences}")
        # print(f"Sample: {sample_size}")
        classes = o.get('class', [])
        for c in classes:
            #print(f"- Class: {c.get('k')}")
            bounds = c.get('bounds', [])
            occ = occurrences[int(c.get('k'))-1]
            #print(f"  - Occurrences for class {c.get('k')}: {occ}")
            for b in bounds:
                #print(f"  - bound: {b}")
                id = int(b.get('id'))
                t = occ[id]
                #print(f"  - Bounds for measure={id}: min={b.get('min')}, max={b.get('max')}, current forecast={t}")
                if t < b.get('min') or t > b.get('max'):
                    ##print(f"    - Measure {id} is outside the bounds!")
                    return True, {"Class": c.get('k'), "Measure": id, "Bounds": {"min": b.get('min'), "max": b.get('max'), "current": t}}
                    
    return False, {}

######################
# Caricamento regole di validazione
evidenze = {}
with open('../collector_results.json') as file_evidenze:
        evidenze = json.load(file_evidenze)
        
# Popolazione        
data_file_name = "../data/estrazioni_2026_2006_copy.csv"
clib = CommonLib(data_file_name)
history_data = clib.load_data() # Carica le estrazioni da un file CSV

ostats = OccurencesStats()
o_limits = get_occurrences_stats(evidenze)

# estrazioni di prova (residuo gennaio 2026)
# 20260131;2;6;7;33;37;80
# 20260130;32;33;39;40;52;86
# 20260129;29;30;34;56;66;80
# 20260127;11;19;27;31;54;84
# 20260124;22;37;55;61;68;71
# 20260123;2;6;11;19;88;90
# 20260122;2;30;52;56;57;78
p = [0] * 7
p [0] = [2,30,52,56,57,78]
p [1] = [2,6,11,19,88,90]
p [2] = [22,37,55,61,68,71]
p [3] = [11,19,27,31,54,84]
p [4] = [29,30,34,56,66,80]
p [5] = [32,33,39,40,52,86]
p [6] = [2,6,7,33,37,80]

for i in range(10):
    base = clib.get_forecast()
    #print("Forecast extraction:", base)
    base_combinations = [list(combinations(base, k)) for k in range(1,7)]
    scarto, log_scarto = check_occurrences(base_combinations, history_data, o_limits)
    if not scarto:
        print(i, base, "At least one measure is outside the bounds.", log_scarto)
    else:
        print(i, base, "All measures are within the bounds.") 
        break
    


0 [ 1  6  8 15 50 79] All measures are within the bounds.
